In [ ]:
# ============================================
# CELL 1: Environment Setup & GPU Configuration
# ============================================

# Check CUDA and install dependencies
import subprocess
import sys

def install_packages():
    packages = [
        "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121",
        "transformers",
        "datasets",
        "peft",
        "trl",
        "bitsandbytes",
        "accelerate",
        "sentencepiece",
        "protobuf",
        "evaluate",
        "rouge-score",
        "wandb"
    ]
    
    for package in packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

install_packages()


In [ ]:
# ============================================
# CELL 2: Imports and Configuration
# ============================================

import os
import gc
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from datasets import Dataset, DatasetDict, load_dataset, concatenate_datasets
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer
from dotenv import load_dotenv
from huggingface_hub import login

# Suppress warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

# GPU Optimizations
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Configuration
MODEL_NAME = "codellama/CodeLlama-7B-Instruct-hf"
OUTPUT_DIR = "./codellama-cobol-python"
HF_REPO_ID = "your-username/codellama-cobol-python"  # Change this

TRAINING_CONFIG = {
    "max_seq_length": 2048,
    "batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "num_epochs": 3,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "use_flash_attention": True,
    "max_samples": 10000  # Limit dataset size for faster training
}


In [ ]:
# ============================================
# CELL 3: Login to Hugging Face
# ============================================

load_dotenv()
hf_token = os.getenv("HUGGINGFACE_TOKEN")
if hf_token:
    login(token=hf_token)
    print("✅ Logged into Hugging Face Hub")
else:
    print("⚠️ No HF token found, will save locally only")


In [ ]:
# ============================================
# CELL 4: Dataset Preparation
# ============================================

def prepare_dataset():
    """Load and prepare datasets for COBOL to Python conversion"""
    
    print("📚 Loading datasets...")
    
    # Load MainframeBench dataset (COBOL with descriptions)
    try:
        mainframe = load_dataset("Fsoft-AIC/MainframeBench", "COBOL_code_summarization", split="train")
        mainframe = mainframe.select(range(min(5000, len(mainframe))))
        print(f"  ✓ MainframeBench: {len(mainframe)} samples")
    except:
        print("  ⚠️ Could not load MainframeBench, skipping...")
        mainframe = None
    
    # Load The Stack datasets
    try:
        stack_cobol = load_dataset("bigcode/the-stack", data_dir="data/cobol", split="train[:1000]")
        stack_python = load_dataset("bigcode/the-stack", data_dir="data/python", split="train[:1000]")
        print(f"  ✓ The Stack COBOL: {len(stack_cobol)} samples")
        print(f"  ✓ The Stack Python: {len(stack_python)} samples")
    except:
        print("  ⚠️ Could not load The Stack, skipping...")
        stack_cobol, stack_python = None, None
    
    # Load Python dataset
    try:
        python_dataset = load_dataset("jtatman/python-code-dataset-500k", split="train[:2000]")
        print(f"  ✓ Python dataset: {len(python_dataset)} samples")
    except:
        print("  ⚠️ Could not load Python dataset, skipping...")
        python_dataset = None
    
    # Format datasets into instruction format
    formatted_datasets = []
    
    # Format MainframeBench
    if mainframe:
        def format_mainframe(example):
            text = f"""### Instruction:
Convert the following COBOL code to Python:

### COBOL:
```cobol
{example['code']}
# Converted Python code
{example.get('summary', '# Automatically convert COBOL to Python')}
```"""
            return {"text": text}
        
        mainframe_formatted = mainframe.map(format_mainframe, remove_columns=mainframe.column_names)
        formatted_datasets.append(mainframe_formatted)
    
    # Format The Stack COBOL-Python pairs
    if stack_cobol and stack_python:
        min_len = min(len(stack_cobol), len(stack_python))
        cobol_pairs = Dataset.from_dict({
            "cobol": stack_cobol["content"][:min_len],
            "python": stack_python["content"][:min_len]
        })
        
        def format_pair(example):
            text = f"""### Instruction:
Convert the following COBOL code to Python:

### COBOL:
```cobol
{example['cobol'][:1500]}
{example['python'][:1500]}
```"""
            return {"text": text}
        
        pairs_formatted = cobol_pairs.map(format_pair)
        formatted_datasets.append(pairs_formatted)
    
    # Combine all datasets
    if formatted_datasets:
        combined = concatenate_datasets(formatted_datasets)
        print(f"\n✅ Combined dataset: {len(combined)} samples")
        
        # Limit dataset size if specified
        if TRAINING_CONFIG.get("max_samples"):
            combined = combined.select(range(min(len(combined), TRAINING_CONFIG["max_samples"])))
        
        # Split into train/validation/test
        split = combined.train_test_split(test_size=0.1, seed=42)
        eval_test = split["test"].train_test_split(test_size=0.5, seed=42)
        
        dataset_dict = DatasetDict({
            "train": split["train"],
            "validation": eval_test["train"],
            "test": eval_test["test"]
        })
        
        print(f"  Train: {len(dataset_dict['train'])} samples")
        print(f"  Validation: {len(dataset_dict['validation'])} samples")
        print(f"  Test: {len(dataset_dict['test'])} samples")
        
        return dataset_dict
    else:
        raise ValueError("No datasets were successfully loaded!")

In [ ]:
# ============================================
# CELL 5: Model and Tokenizer Loading
# ============================================

def load_model_and_tokenizer():
    """Load CodeLlama with 4-bit quantization for efficient fine-tuning"""
    
    # Quantization config
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    
    # Load tokenizer
    print(f"🔧 Loading tokenizer: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        padding_side="right",
        add_eos_token=True
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load model
    print(f"🔧 Loading model with 4-bit quantization...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2" if TRAINING_CONFIG.get("use_flash_attention", True) else "eager"
    )
    
    # Prepare for k-bit training
    model = prepare_model_for_kbit_training(model)
    
    # Apply LoRA
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=TRAINING_CONFIG["lora_r"],
        lora_alpha=TRAINING_CONFIG["lora_alpha"],
        lora_dropout=TRAINING_CONFIG["lora_dropout"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        bias="none",
    )
    
    model = get_peft_model(model, lora_config)
    
    # Print trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n📊 Model Statistics:")
    print(f"  Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
    print(f"  Total parameters: {total_params:,}")
    
    return model, tokenizer


In [ ]:
# ============================================
# CELL 6: Training Setup
# ============================================

def setup_training_args():
    """Configure training arguments"""
    return TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=TRAINING_CONFIG["num_epochs"],
        per_device_train_batch_size=TRAINING_CONFIG["batch_size"],
        per_device_eval_batch_size=TRAINING_CONFIG["batch_size"],
        gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        bf16=True,
        tf32=True,
        dataloader_pin_memory=True,
        gradient_checkpointing=True,
        optim="adamw_torch_fused",
        max_grad_norm=1.0,
        logging_steps=25,
        save_steps=500,
        eval_steps=500,
        evaluation_strategy="steps",
        save_total_limit=2,
        remove_unused_columns=False,
        report_to="none",  # Set to "wandb" if you want to use wandb
        run_name="codellama-cobol-python",
    )


In [ ]:
# ============================================
# CELL 7: Training Execution
# ============================================

dataset_dict = prepare_dataset()
model, tokenizer = load_model_and_tokenizer()

training_args = setup_training_args()

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["validation"],
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=TRAINING_CONFIG["max_seq_length"],
    packing=False,
)

# Start training
print("\n🚀 Starting training...")
trainer.train()

# Save the model
print("\n💾 Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")


In [ ]:
# ============================================
# CELL 8: Evaluation
# ============================================

def evaluate_model(trainer, dataset_dict):
    """Evaluate the fine-tuned model"""
    print("\n🔍 Evaluating model...")
    
    test_samples = dataset_dict["test"].select(range(min(5, len(dataset_dict["test"]))))
    
    pipe = pipeline(
        "text-generation",
        model=trainer.model,
        tokenizer=trainer.tokenizer,
        device_map="auto"
    )
    
    results = []
    
    for sample in test_samples:
        prompt = sample["text"].split("### Response:")[0] + "### Response:\n```python\n"
        
        response = pipe(
            prompt,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False,
        )
        
        generated = response[0]["generated_text"]
        python_code = generated.split("### Response:")[-1].strip()
        
        results.append({
            "prompt": prompt[:200],
            "generated": python_code[:500]
        })
    
    # Display results
    for i, result in enumerate(results):
        print(f"\n--- Sample {i+1} ---")
        print(f"Generated Python:\n{result['generated']}\n")
    
    return results

evaluation_results = evaluate_model(trainer, dataset_dict)


In [ ]:
# ============================================
# CELL 9: Merge and Save Final Model
# ============================================

from peft import PeftModel

print("\n🔧 Merging LoRA weights with base model...")

# Reload base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

# Load adapters and merge
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()

# Save merged model
merged_model.save_pretrained(OUTPUT_DIR + "-merged")
tokenizer.save_pretrained(OUTPUT_DIR + "-merged")
print(f"✅ Merged model saved to {OUTPUT_DIR}-merged")


In [ ]:
# ============================================
# CELL 10: Push to Hugging Face Hub (Optional)
# ============================================

if hf_token and HF_REPO_ID != "your-username/codellama-cobol-python":
    print("\n📤 Pushing model to Hugging Face Hub...")
    
    merged_model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    
    print(f"✅ Model pushed to https://huggingface.co/{HF_REPO_ID}")
else:
    print("\n⚠️ Skipping upload to Hub (no HF token or repo ID not configured)")


In [ ]:
# ============================================
# CELL 11: Test Inference
# ============================================

print("\n🧪 Testing model with a sample COBOL program...")

cobol_sample = """
IDENTIFICATION DIVISION.
PROGRAM-ID. CALCULATE.
DATA DIVISION.
WORKING-STORAGE SECTION.
01 NUM1 PIC 9(3) VALUE 10.
01 NUM2 PIC 9(3) VALUE 20.
01 RESULT PIC 9(3).
PROCEDURE DIVISION.
    ADD NUM1 TO NUM2 GIVING RESULT.
    DISPLAY "RESULT: " RESULT.
    STOP RUN.
"""

prompt = f"""### Instruction:
Convert the following COBOL code to Python:

### COBOL:
```cobol
{cobol_sample}

In [ ]:
"""

pipe = pipeline("text-generation", model=merged_model, tokenizer=tokenizer, device_map="auto")
response = pipe(prompt, max_new_tokens=256, temperature=0.1, do_sample=False)

generated_code = response[0]["generated_text"].split("### Response:")[-1].strip()
print(f"\nGenerated Python Code:\n{generated_code}")

print("\n✅ Fine-tuning pipeline complete!")